# Causal Inference Crash Course:
## Regression Discontinuity Design
Julian Hsu

## Overview
* **Regression discontinuity (RDD)**: a local natural experiment hiding at a cutoff.
* Earlier decks controlled for confounders (unconfoundedness). RDD instead assumes only **continuity** at the cutoff.
* We cover:
    * The big idea and the *local* estimand
    * Sharp vs. fuzzy designs
    * Estimation: one regression, plus three practical questions
    * Validating the design
    * A simulated worked example
* Assumes the earlier decks: potential outcomes, OLS, inference.

## A threshold-based policy
* A **promotion** ($W_i$) turns on once an **eligibility score** ($X_i$) crosses a cutoff $c$.
    * e.g. a spend threshold for free shipping, a credit-score cutoff for an offer.
* We want its effect on **next-quarter purchases** ($Y_i$).
* Customers just below and just above $c$ are nearly identical — only the promotion differs.
* That threshold is the experiment.

## The Big Idea
![Image](Figures/RD_Figure_0.png)
* At the cutoff, treatment switches on (left) — and the outcome jumps with it (right).
* The jump in $Y$ *is* the treatment effect.

## Why not just run a regression?
* The simple linear model $Y_i = \alpha + \tau W_i + \beta X_i + \epsilon_i$ uses **all** the data and assumes one straight line.
* But $X_i$ *perfectly determines* $W_i$: no overlap, so $\tau$ leans entirely on that functional form.
* RDD trusts the comparison **only at the cutoff**:
$$ \tau_{RD} = \lim_{x \downarrow c} E[Y_i \mid X_i=x] \;-\; \lim_{x \uparrow c} E[Y_i \mid X_i=x] $$

## The assumption: continuity
* $E[Y_i(0)\mid X]$ and $E[Y_i(1)\mid X]$ do not jump at $c$.
* So absent treatment, nothing else changes at the cutoff — only $W_i$ can make $Y$ jump.
* Unlike unconfoundedness, we needn't observe confounders — only that they move **smoothly** through $c$.
* The biggest threat: **manipulation** of $X_i$ to land just past $c$.
* Also audit for a **shared cutoff**: if another program keys on the same threshold, the jump bundles both effects.

## RDD estimates a *local* effect
* $\tau_{RD}$ is the effect **at the cutoff**, for units with $X_i \approx c$.
* Credible internally; limited externally — it needn't hold far from $c$.
* So you need data near $c$, and your answer is *about* $c$.
* The flip side: when the decision *is* the threshold — "should we move the cutoff?" — the effect at $c$ is exactly the number you need.

# Sharp vs. Fuzzy
Does treatment switch on completely at the cutoff, or only partly?

## Sharp: treatment jumps 0 → 1
![Image](Figures/RD_Figure_0.png)
* Everyone above the cutoff is treated, no one below: $W_i = 1\{X_i \ge c\}$.
* The jump in $Y$ is the treatment effect, directly.

## Fuzzy: treatment jumps only partway
![Image](Figures/RD_Figure_5.png)
* Same two pictures — but now crossing $c$ only *raises the chance* of treatment (some below get it, some above don't).
* The outcome jump is **diluted**: only the extra treated units contribute.

## Fuzzy RD: rescale the diluted jump
* Divide the outcome jump by the treatment jump — both read straight off the two panels:
$$ \tau_{FRD} = \frac{\text{jump in } E[Y\mid X] \text{ at } c}{\text{jump in } E[W\mid X] \text{ at } c} $$
* This is IV, with $1\{X_i \ge c\}$ as the instrument — a LATE for compliers at the cutoff.
* Being IV, it carries the usual extra assumptions: **exclusion** (crossing $c$ moves $Y$ only through $W$) and **monotonicity** (crossing $c$ pushes no one *out* of treatment).
* Sharp is the special case where the denominator is 1.
* Estimation and a worked demo: [**Appendix — fuzzy RD in practice**](#/appendix-fuzzy-rd-in-practice).

# Estimation
One OLS regression, near the cutoff.

## The RD regression
* Center the score at the cutoff and run **one OLS**, fully interacting treatment with the running variable:
$$ Y_i = \alpha + \tau W_i + \beta (X_i - c) + \gamma\, W_i (X_i - c) + \epsilon_i $$
* Each piece has a job:
    * $\beta$: the slope below the cutoff
    * $\beta + \gamma$: the slope above — the interaction lets the sides differ
    * $\tau$: the gap between the two lines *at* $X_i = c$ — the treatment effect
* Fit it using only observations near the cutoff ($|X_i - c| \le h$).
    * (In practice the window is usually kernel-weighted too — [**Appendix — kernel weights and RBC detail**](#/appendix-kernel-weights-and-rbc-detail).)

## What it looks like
![Image](Figures/RD_Figure_6.png)
* One regression, drawn: a line on each side, and $\hat\tau$ is the gap at the cutoff.

## Question 1: what if the relationship is curved?
* If $E[Y\mid X]$ bends inside the window, straight lines mis-measure the jump.
* Natural fix: add polynomial terms, still interacted with $W_i$:
$$ Y_i = \alpha + \tau W_i + \sum_{k=1}^{p} \left[\beta_k (X_i-c)^k + \gamma_k W_i (X_i-c)^k\right] + \epsilon_i $$
* Keep $p$ **low** (1 or 2). Higher orders chase noise — next slide.

## Why not a high-order polynomial on all the data?
![Image](Figures/RD_Figure_1.png)
* Far-away data then controls the fit at the cutoff, and $\hat\tau$ **swings with the order** (Gelman & Imbens 2019).
* Curvature is better handled by *narrowing the window* than by raising the order.

## Question 2: how wide a window? Bias vs. variance
![Image](Figures/RD_Figure_2.png)
* **Narrow** $h$: a straight line is locally right (*low bias*), but few observations (*high variance*).
* **Wide** $h$: more observations (*low variance*), but the line is more wrong (*high bias*).
* The **MSE-optimal** bandwidth (CCT 2014) balances the two — a data-driven answer, not a hand-picked one.

## Question 3: can I use the usual confidence intervals?
* No — and the reason is the bandwidth choice you just made.
* The MSE-optimal $h$ *accepts some bias on purpose* (that's the trade-off). So $\hat\tau$ is centered slightly off the truth.
* A textbook CI is built around $\hat\tau$ — centered on a biased point, it covers the truth **less** than 95% of the time.
* **Robust bias-corrected (RBC)** inference: estimate the bias, subtract it, and widen the CI for the extra noise of estimating it. That is `rdrobust`'s **Robust** row.

# Validating the design
Continuity has testable implications.

## Continuity is testable
* If continuity holds:
    1. the density of $X_i$ doesn't jump at $c$ (no manipulation);
    2. predetermined covariates don't jump at $c$;
    3. fake cutoffs show no effect;
    4. the estimate survives dropping points at the cutoff (donut) and changing $h$.
* These are tests of *implications*: failing one refutes the design, but passing all four earns credibility, not proof.

## Test 1: no manipulation
* If customers can nudge their score, the ones just above $c$ *chose* to be there — not comparable to those just below.
* The tell is a **pile-up** just above $c$:
![Image](Figures/RD_Figure_3.png)
* Formal test: is the density of $X_i$ continuous at $c$? (McCrary 2008; `rddensity`.)
* Nuance: customers may *influence* their score — that alone is fine. The design breaks only under **precise** control over landing just past $c$ (Lee 2008).

## Test 2: covariates don't jump
* Just-left and just-right customers should look identical *before* treatment.
* Check it directly: rerun the **same RD regression** with a pre-treatment covariate as the outcome. The "effect" should be zero.
![Image](Figures/RD_Figure_4.png)

## Test 3: placebo cutoffs
* Move the cutoff to a value where nothing actually changes — say $c=40$ or $c=60$ — and re-run the RD.
* No policy switches there, so the true jump is zero.
* Fit each placebo on one side of the real cutoff only (e.g. only $X_i<50$ data for $c=40$), so the true jump at $50$ can't contaminate it.
* If you still find a "jump", your specification finds jumps everywhere, and the one at $c$ means nothing.

## Test 4: the donut
* Manipulators land **right at the cutoff** — the observations closest to $c$ are the least trustworthy.
* The donut test: drop points within $\pm r$ of $c$, refit, and widen the hole. A clean design barely moves; a sorted one moves a lot.
* Worked example with figures: [**Appendix — the donut test**](#/appendix-the-donut-test).

## Test 5: bandwidth sensitivity
* Re-estimate across a range of bandwidths (the bias–variance figure already shows this curve).
* A credible estimate is **stable** over reasonable $h$ — not a knife-edge result that appears at exactly one bandwidth.

# Worked example
Simulated data, so we know the answer ($\tau = 5$).

## Two scenarios
* Score $X_i\in[0,100]$, cutoff $c=50$, promotion $W_i=1\{X_i\ge c\}$, outcome $Y_i$ = next-quarter purchases, true $\tau=5$.
1. **Clean** — continuity holds and $E[Y\mid X]$ is a straight line.
2. **Nonlinear** — continuity still holds, but $E[Y\mid X]$ is curved. Stress-tests the functional form.
* What if continuity *fails*? See [**Appendix — the donut test**](#/appendix-the-donut-test).
* Companion notebook with all code [here](https://github.com/shoepaladin/causalinference_crashcourse/blob/main/Notebooks/7%20Regression%20Discontinuity.ipynb).

## Four models
1. **Naive difference**: mean $Y$ above the cutoff minus mean $Y$ below. Ignores $X_i$ entirely.
2. **Global polynomial**: the RD regression on *all* the data, with 4th-order $(X_i-c)$ terms, interacted.
3. **Local linear**: the RD regression from the estimation section — linear, within $h=8$ of the cutoff, triangular weights.
4. **`rdrobust`**: local linear with the MSE-optimal bandwidth and robust bias-corrected confidence intervals.

## Bias by model and scenario
* Mean bias $(\hat\tau-\tau)$ over 500 simulations; $\tau=5$.

| Model | Clean | Nonlinear |
|:---|:---:|:---:|
| Naive difference | +2.28 | +5.33 |
| Global polynomial | +0.01 | -0.04 |
| Local linear | +0.02 | +0.05 |
| `rdrobust` (RBC) | +0.00 | -0.06 |

* The naive difference is biased even in the clean case — it compares high scorers to low scorers, not the two sides of the cutoff.
* All three RD models handle the curvature.

## Coverage
* How often the 95% confidence interval contains the truth:

| Model | Clean | Nonlinear |
|:---|:---:|:---:|
| Local linear (naive SE) | 0.95 | 0.94 |
| `rdrobust` (RBC) | 0.94 | 0.95 |

* With a valid design, the intervals deliver their nominal 95%.
* When the design is broken, they don't — see [**Appendix — the donut test**](#/appendix-the-donut-test).

## Conclusion
* RDD is a credible **local experiment** at a cutoff.
* Pros: transparent, a *testable* assumption, strong visuals.
* Cons: only a **local** effect, needs data near $c$, and it dies under manipulation.
* Reach for it when a real decision turns on a threshold — then run the checks.

# Appendix Slides

## Appendix: the donut test
* The scenario, in our retail example: customers just below the cutoff nudge their score above it — retry a transaction, time a purchase — and the ones who bother are the **higher-spending** ones.
* The just-above group is no longer comparable to the just-below group: **continuity is broken**, and every estimator inherits the bias.
* The sorters sit **right at the cutoff**. The donut test exploits that: drop the points nearest $c$ and see whether the estimate survives.

## The donut in action
![Image](Figures/RD_Figure_8.png)
* Left: sorting drags the fit up — $\hat\tau = 6.7$ vs. a truth of 5. Note the tell: a dip just below the cutoff, a bulge just above.
* Right: drop the shaded $\pm 6$ hole and refit from the outside — $\hat\tau = 5.0$.

## Widening the donut
![Image](Figures/RD_Figure_7.png)
* Repeat over radii. Clean design: flat. Sorted design: the estimate moves a lot, drifting toward the truth as the sorters are dropped.
* The *movement* is the diagnostic — a stable path is evidence the design is clean.

## No estimator survives manipulation
* The same 500-simulation study as the worked example, under the sorting scenario:

| Model | Bias | 95% CI coverage |
|:---|:---:|:---:|
| Naive difference | +2.60 | &mdash; |
| Global polynomial | +1.99 | &mdash; |
| Local linear | +1.73 | 0.00 |
| `rdrobust` (RBC) | +1.89 | 0.00 |

* Every model is biased and the intervals essentially *never* contain the truth.
* The donut **diagnoses** the problem; it does not license the original estimate. Validation comes first.

## Appendix: fuzzy RD in practice
* In the field, eligibility rarely equals take-up: crossing $c$ *unlocks* the promotion, but not everyone redeems it. Most industry threshold designs are fuzzy.
* Estimation is the same local machinery run twice — the jump in $Y$ (intent-to-treat) divided by the jump in $W$ (first stage). `rdrobust(y, x, fuzzy=w)` does both in one call.
* **Report the first stage.** A small jump in take-up is a weak instrument: intervals widen fast and the estimate turns fragile.
* Validation is unchanged — the density and covariate tests involve only $X$, so they don't care that treatment is fuzzy.
* Worked demo (hand-rolled Wald ratio vs. `rdrobust`) in the companion notebook.

## Appendix: covariates and power
* **Covariates.** Predetermined covariates can't rescue a broken design; what they buy is *precision* (Calonico, Cattaneo, Farrell, Titiunik 2019; `rdrobust(covs=)`). In the companion notebook the robust CI narrows by about a quarter.
* If $\hat\tau$ *moves* when covariates enter, treat it as a red flag — smooth covariates shouldn't matter, so something is jumping at $c$.
* **Power.** Only observations near $c$ carry information: in the notebook demo the RD interval is $\sim$3&times; wider than an RCT on the *same* customers — roughly 10&times; the sample for equal precision.
* Budget before you commit: `rdpower` does MDE and sample-size calculations for RD designs.

## Appendix: cautions from the field
* **RD in time.** A launch date is not a cutoff: the same units sit on both sides, and trends, anticipation and autocorrelation — not continuity — drive the comparison (Hausman & Rapson 2018). Reach for event-study / DiD tools instead.
* **Extrapolation.** $\tau_{RD}$ is the effect at $c$. Costing a full rollout (or shutdown) means extrapolating away from the cutoff — that takes extra assumptions, and they should be stated, not smuggled.
* **Shared cutoffs.** Round numbers attract policies: if free shipping *and* a loyalty tier both switch at \$50, the jump prices the bundle. Audit everything that changes at $c$.

## Appendix: the kink design (RKD)
* Sometimes treatment *intensity* is continuous at $c$ but its **slope** changes (e.g. a benefit formula).
* RKD reads the effect from the change in **slope** of $Y$, not a jump in level. Same tools (`rdrobust`, `deriv=1`).

## Appendix: extensions
* **Multiple / multi-dimensional cutoffs** — e.g. geographic RD.
* **Discrete running variables** — use the local randomization framework (Cattaneo, Idrobo, Titiunik 2024).
    * With few distinct score values, "close to $c$" is coarse: see also honest CIs (Kolesár & Rothe 2018) and `rdrobust`'s mass-point handling.
* **Local randomization** — treat a window around $c$ as an experiment (`rdlocrand`).

## Appendix: kernel weights and RBC detail
* The RD regression is usually fit by weighted least squares, down-weighting points far from the cutoff:
$$ \min_{\alpha,\tau,\beta,\gamma}\; \sum_i K\!\left(\frac{X_i-c}{h}\right)\left(Y_i - \alpha - \tau W_i - \beta(X_i-c) - \gamma W_i (X_i-c)\right)^2 $$
* A triangular kernel $K$ is MSE-optimal at a boundary; a uniform kernel = plain OLS in the window.
* RBC estimates the leading bias with a second (pilot) bandwidth $b \ge h$, subtracts it, and inflates the variance accordingly.

## Literature Review of Related Papers
* **Foundational**
    * Thistlethwaite & Campbell (1960). "Regression-Discontinuity Analysis." *J. Educational Psychology.*
    * Hahn, Todd, Van der Klaauw (2001). "Identification and Estimation of Treatment Effects with a RD Design." *Econometrica.* [link](https://www.jstor.org/stable/2692190)
    * Lee (2008). "Randomized experiments from non-random selection in U.S. House elections." [link](https://www.sciencedirect.com/science/article/abs/pii/S0304407607001121)
    * Imbens & Lemieux (2008). "Regression discontinuity designs: A guide to practice." [link](https://www.nber.org/papers/w13039)
    * Lee & Lemieux (2010). "Regression Discontinuity Designs in Economics." *J. Economic Literature.* [link](https://www.aeaweb.org/articles?id=10.1257/jel.48.2.281)
* **Estimation & inference**
    * Calonico, Cattaneo, Titiunik (2014). "Robust Nonparametric Confidence Intervals for RD Designs." *Econometrica.* [link](https://onlinelibrary.wiley.com/doi/10.3982/ECTA11757)
    * Calonico, Cattaneo, Farrell (2020). "Optimal bandwidth choice for robust bias-corrected inference in RD designs." [link](https://academic.oup.com/ectj/article/23/2/192/5698687)
    * Calonico, Cattaneo, Farrell, Titiunik (2019). "Regression Discontinuity Designs Using Covariates." *REStat.* [link](https://rdpackages.github.io/references/Calonico-Cattaneo-Farrell-Titiunik_2019_RESTAT.pdf)
    * Card, Lee, Pei, Weber (2015). "Inference on Causal Effects in a Generalized Regression Kink Design." *Econometrica.* [link](https://onlinelibrary.wiley.com/doi/10.3982/ECTA11224)
    * Kolesár & Rothe (2018). "Inference in RD Designs with a Discrete Running Variable." *AER.* [link](https://www.aeaweb.org/articles?id=10.1257/aer.20160945)
    * Gelman & Imbens (2019). "Why High-Order Polynomials Should Not Be Used in RD Designs." [link](https://www.tandfonline.com/doi/full/10.1080/07350015.2017.1366909)
* **Practical guides**
    * Cattaneo, Idrobo, Titiunik (2020, 2024). "A Practical Introduction to RD Designs: Foundations / Extensions." Cambridge. [link](https://rdpackages.github.io/references/Cattaneo-Idrobo-Titiunik_2020_CUP.pdf)
    * Hausman & Rapson (2018). "Regression Discontinuity in Time: Considerations for Empirical Applications." *Annu. Rev. Resource Econ.* [link](https://www.annualreviews.org/doi/10.1146/annurev-resource-121517-033306)
* **Validation & software**
    * McCrary (2008). "Manipulation of the running variable in the RD design: A density test." [link](https://www.sciencedirect.com/science/article/abs/pii/S0304407607001133)
    * Cattaneo, Jansson, Ma (2020). "Simple Local Polynomial Density Estimators." (`rddensity`) [link](https://nppackages.github.io/references/Cattaneo-Jansson-Ma_2020_JASA.pdf)
    * Cattaneo, Titiunik, Vazquez-Bare (2019). "Power Calculations for Regression-Discontinuity Designs." *Stata Journal.* (`rdpower`) [link](https://rdpackages.github.io/rdpower/)
    * Software: `rdrobust`, `rddensity`, `rdlocrand`, `rdpower` &mdash; [rdpackages.github.io](https://rdpackages.github.io)

In [ ]:
# --- Figure generation (run this notebook to regenerate Figures/RD_Figure_*.png) ---
# All figures below are produced from a simulated retail RD design.
import os
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(20260702)
CUT, TAU = 50.0, 5.0
ACCENT, ACCENT2, CTRLC = "#0e7490", "#0f766e", "#94a3b8"
FIGDIR = os.path.join(os.getcwd(), "Figures")
os.makedirs(FIGDIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 11, "axes.grid": True, "grid.alpha": .25})


def baseline(x, shape="linear"):
    xc = (x - CUT) / 10.0
    if shape == "linear":
        return 10.0 + 0.8 * xc
    return 10.0 + 0.8 * xc + 2.6 * np.sin(0.9 * xc)  # "nonlinear": high curvature


def gen(n=2000, shape="linear", manipulate=False, fuzzy=False, seed=None):
    rng = np.random.default_rng(seed) if seed is not None else RNG
    x = np.clip(rng.normal(CUT, 18, n), 0, 100)
    ability = rng.normal(0, 1, n)                      # unobserved outcome driver
    if manipulate:                                     # high-ability units sort above c
        below = (x > CUT - 6) & (x < CUT)
        move = below & (ability > 0.2) & (rng.random(n) < 0.75)
        x[move] = CUT + rng.uniform(0, 4, move.sum())
    if fuzzy:
        w = (rng.random(n) < np.where(x >= CUT, 0.75, 0.20)).astype(float)
    else:
        w = (x >= CUT).astype(float)
    y = baseline(x, shape) + TAU * w + 2.0 * ability + rng.normal(0, 1.5, n)
    z = 0.5 + 0.03 * (x - CUT) + 0.8 * ability + rng.normal(0, 0.6, n)  # predetermined covariate
    return x, w, y, z


def naive(x, w, y):
    return y[w == 1].mean() - y[w == 0].mean()


def global_poly(x, w, y, order=4):
    xc = x - CUT
    feats = [np.ones_like(xc), w]
    for k in range(1, order + 1):
        feats += [xc ** k, w * xc ** k]
    beta, *_ = np.linalg.lstsq(np.column_stack(feats), y, rcond=None)
    return beta[1]


def local_linear(x, w, y, h=8.0, return_se=False):
    """The RD regression: one interacted OLS within |X-c|<=h, triangular weights.
    (Numerically identical to fitting each side separately.)"""
    xc = x - CUT
    ints, ses = {}, {}
    for side, mask in (("r", w == 1), ("l", w == 0)):
        xs, ys = xc[mask], y[mask]
        sel = np.abs(xs) <= h
        xs, ys = xs[sel], ys[sel]
        wgt = np.maximum(1 - np.abs(xs / h), 0)                     # triangular kernel
        X = np.column_stack([np.ones_like(xs), xs]); W = np.diag(wgt)
        XtW = X.T @ W
        beta = np.linalg.solve(XtW @ X, XtW @ ys)
        ints[side] = beta[0]
        resid = ys - X @ beta
        s2 = (wgt * resid ** 2).sum() / max(wgt.sum() - 2, 1)
        cov = np.linalg.inv(XtW @ X) @ (XtW @ W @ X) @ np.linalg.inv(XtW @ X) * s2
        ses[side] = np.sqrt(max(cov[0, 0], 0))
    tau = ints["r"] - ints["l"]
    return (tau, np.sqrt(ses["r"] ** 2 + ses["l"] ** 2)) if return_se else tau


def _save(fig, name):
    fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, name), bbox_inches="tight"); plt.close(fig)


def _band(ax):
    ax.axvline(CUT, color="#334155", ls="--", lw=1.2)


def _binned(ax, x, v, color=ACCENT):
    # bins never straddle the cutoff: separate grids on each side
    for lo, hi, col in ((0, CUT, CTRLC), (CUT, 100, color)):
        bins = np.linspace(lo, hi, 14)
        mask = (x >= lo) & (x < hi)
        idx = np.digitize(x, bins)
        bx = [x[mask & (idx == b)].mean() for b in range(1, len(bins)) if (mask & (idx == b)).any()]
        bv = [v[mask & (idx == b)].mean() for b in range(1, len(bins)) if (mask & (idx == b)).any()]
        ax.scatter(bx, bv, s=26, color=col, edgecolor="white", zorder=3)


def _two_panel(x, w, y, fname, w_title, y_title):
    """Shared format for sharp (Fig 0) and fuzzy (Fig 5): W jump left, Y jump right."""
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.3))
    ax = axes[0]
    _binned(ax, x, w)
    _band(ax); ax.set_ylim(-.05, 1.05)
    ax.set_xlabel("Eligibility score  X"); ax.set_ylabel("Share receiving promotion  W")
    ax.set_title(w_title, fontsize=10)
    ax = axes[1]
    _binned(ax, x, y)
    for mask, col in ((x < CUT, "#0f172a"), (x >= CUT, ACCENT2)):
        xs = np.sort(x[mask]); ax.plot(xs, np.polyval(np.polyfit(x[mask], y[mask], 1), xs), color=col, lw=2)
    _band(ax)
    ax.set_xlabel("Eligibility score  X"); ax.set_ylabel("Next-quarter purchases  Y ($)")
    ax.set_title(y_title, fontsize=10)
    _save(fig, fname)

In [ ]:
# Figure 0 — sharp design: treatment jumps 0 -> 1 (left), outcome jumps with it (right)
x, w, y, z = gen(3000, "linear", seed=7)
_two_panel(x, w, y, "RD_Figure_0.png",
           "Treatment jumps 0 → 1 at the cutoff",
           "…and the outcome jumps with it")

In [ ]:
# Figure 5 — fuzzy design, SAME format: treatment jumps only partway, outcome jump diluted
x, w, y, z = gen(4000, "linear", fuzzy=True, seed=4)
_two_panel(x, w, y, "RD_Figure_5.png",
           "Treatment jumps, but not 0 → 1",
           "…so the outcome jump is diluted")

In [ ]:
# Figure 6 — the RD regression, drawn: binned means + a line on each side
x, w, y, z = gen(3000, "linear", seed=3)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
_binned(ax, x, y)
for mask, col in ((x < CUT, "#0f172a"), (x >= CUT, ACCENT2)):
    xs = np.sort(x[mask]); ax.plot(xs, np.polyval(np.polyfit(x[mask], y[mask], 1), xs), color=col, lw=2)
_band(ax)
ax.set_xlabel("Eligibility score  X"); ax.set_ylabel("Binned mean of  Y")
ax.set_title("The RD regression: a line each side, $\\hat\\tau$ = the gap at $c$", fontsize=10)
_save(fig, "RD_Figure_6.png")

In [ ]:
# Figure 1 — global polynomials are sensitive to order (Gelman & Imbens 2019)
x, w, y, z = gen(2500, "nonlinear", seed=11)
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.4))
ax = axes[0]; ax.scatter(x, y, s=6, color="#cbd5e1", alpha=.5); _band(ax)
gx0 = np.linspace(0, CUT, 200); gx1 = np.linspace(CUT, 100, 200)
for gx, mask in ((gx0, x < CUT), (gx1, x >= CUT)):
    ax.plot(gx, np.polyval(np.polyfit(x[mask], y[mask], 1), gx), color="#f59e0b", lw=2, ls="--")
    ax.plot(gx, np.polyval(np.polyfit(x[mask], y[mask], 6), gx), color="#b91c1c", lw=2)
for gx, mask, col in ((gx0, x < CUT, "#0f172a"), (gx1, x >= CUT, ACCENT2)):
    m = mask & (np.abs(x - CUT) <= 8); gg = gx[np.abs(gx - CUT) <= 8]
    ax.plot(gg, np.polyval(np.polyfit(x[m], y[m], 1), gg), color=col, lw=3)
ax.plot([], [], color="#f59e0b", ls="--", label="global order 1")
ax.plot([], [], color="#b91c1c", label="global order 6")
ax.plot([], [], color=ACCENT2, lw=3, label="local linear")
ax.legend(fontsize=7.5, loc="upper left"); ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title("One dataset, three fits", fontsize=10)
orders = range(1, 8); mean, sd = [], []
for o in orders:
    b = [global_poly(*gen(3000, "nonlinear", seed=300 + r)[:3], o) for r in range(120)]
    mean.append(np.mean(b)); sd.append(np.std(b))
ll = [local_linear(*gen(3000, "nonlinear", seed=300 + r)[:3], 8.0) for r in range(120)]
ax = axes[1]
ax.errorbar(list(orders), mean, yerr=sd, color="#b91c1c", marker="o", ms=4, capsize=3, label="global polynomial")
ax.axhline(np.mean(ll), color=ACCENT2, lw=2, label="local linear")
ax.axhline(TAU, color="#0f172a", ls="--", lw=1.2, label=f"true $\\tau$={TAU:.0f}")
ax.set_xlabel("global polynomial order"); ax.set_ylabel("estimated  $\\hat\\tau$")
ax.set_title("Estimate swings with the order", fontsize=10); ax.legend(fontsize=7.5)
_save(fig, "RD_Figure_1.png")

In [ ]:
# Figure 2 — bandwidth / bias-variance tradeoff
x, w, y, z = gen(4000, "nonlinear", seed=5)
hs = np.arange(3, 31, 1.0); est, lo, hi = [], [], []
for h in hs:
    t, se = local_linear(x, w, y, h, return_se=True)
    est.append(t); lo.append(t - 1.96 * se); hi.append(t + 1.96 * se)
fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.fill_between(hs, lo, hi, color=ACCENT, alpha=.15)
ax.plot(hs, est, color=ACCENT, lw=2, marker="o", ms=3, label="Local-linear $\\hat\\tau$")
ax.axhline(TAU, color="#0f172a", ls="--", lw=1.2, label=f"true $\\tau$={TAU:.0f}")
try:                                   # MSE-optimal bandwidth from rdrobust, if installed
    from rdrobust import rdrobust
    hopt = float(rdrobust(y=y, x=x, c=CUT).bws.loc["h", "left"])
    ax.axvline(hopt, color=ACCENT2, ls=":", lw=1.5, label=f"MSE-opt h≈{hopt:.1f}")
except Exception:
    pass
ax.set_xlabel("bandwidth  h"); ax.set_ylabel("estimated  $\\hat\\tau$"); ax.legend(fontsize=8)
_save(fig, "RD_Figure_2.png")

In [ ]:
# Figure 3 — running-variable density: clean vs. manipulated
xc, *_ = gen(6000, "linear", seed=2)
xm, *_ = gen(6000, "linear", manipulate=True, seed=2)
fig, axes = plt.subplots(1, 2, figsize=(7.6, 2.9), sharey=True)
for ax, data, title in zip(axes, [xc, xm], ["No manipulation", "Sorting at the cutoff"]):
    ax.hist(data[data < CUT], bins=np.linspace(0, CUT, 26), color=CTRLC, alpha=.85)
    ax.hist(data[data >= CUT], bins=np.linspace(CUT, 100, 26), color=ACCENT, alpha=.85)
    _band(ax); ax.set_title(title, fontsize=10); ax.set_xlabel("X")
axes[0].set_ylabel("count")
_save(fig, "RD_Figure_3.png")

In [ ]:
# Figure 4 — covariate continuity placebo (a predetermined covariate should not jump)
x, w, y, z = gen(3000, "linear", seed=9)
fig, ax = plt.subplots(figsize=(6.2, 3.4))
_binned(ax, x, z)
for mask, col in ((x < CUT, "#0f172a"), (x >= CUT, ACCENT2)):
    xs = np.sort(x[mask]); ax.plot(xs, np.polyval(np.polyfit(x[mask], z[mask], 1), xs), color=col, lw=2)
_band(ax)
ax.set_xlabel("Eligibility score  X"); ax.set_ylabel("Predetermined covariate  Z")
ax.set_title("Placebo: a pre-treatment covariate should NOT jump", fontsize=9.5)
_save(fig, "RD_Figure_4.png")

In [ ]:
# Figure 7 — the donut test: estimate vs donut radius, clean vs manipulated
radii = np.arange(0, 6.5, 0.5)
fig, ax = plt.subplots(figsize=(6.4, 3.0))
for man, col, lab in ((False, ACCENT, "clean"), (True, "#b91c1c", "manipulated")):
    est = []
    for r in radii:
        vals = []
        for s in range(30):                      # average over datasets to smooth
            x, w, y, z = gen(4000, "linear", manipulate=man, seed=500 + s)
            keep = np.abs(x - CUT) >= r
            vals.append(local_linear(x[keep], w[keep], y[keep], 8.0 + r))
        est.append(np.mean(vals))
    ax.plot(radii, est, color=col, lw=2, marker="o", ms=4, label=lab)
ax.axhline(TAU, color="#0f172a", ls="--", lw=1.2, label=f"true $\\tau$={TAU:.0f}")
ax.set_xlabel("donut radius (points dropped within $\\pm r$ of the cutoff)")
ax.set_ylabel("estimated  $\\hat\\tau$")
ax.legend(fontsize=8.5)
_save(fig, "RD_Figure_7.png")

In [ ]:
# Figure 8 — the donut in action: manipulated data, fit with vs. without a +-6 hole
R = 6.0
x, w, y, z = gen(6000, "linear", manipulate=True, seed=45)


def _binned_range(ax, x, v, lo, hi, col, n=14):
    bins = np.linspace(lo, hi, n)
    mask = (x >= lo) & (x < hi)
    idx = np.digitize(x, bins)
    bx = [x[mask & (idx == b)].mean() for b in range(1, len(bins)) if (mask & (idx == b)).any()]
    bv = [v[mask & (idx == b)].mean() for b in range(1, len(bins)) if (mask & (idx == b)).any()]
    ax.scatter(bx, bv, s=26, color=col, edgecolor="white", zorder=3)


fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.4), sharey=True)
ax = axes[0]                                     # all data: biased fit
_binned_range(ax, x, y, 0, CUT, CTRLC); _binned_range(ax, x, y, CUT, 100, ACCENT)
for side_mask, col in ((x < CUT, "#0f172a"), (x >= CUT, ACCENT2)):
    m = side_mask & (np.abs(x - CUT) <= 12)
    gx = np.linspace(CUT - 12, CUT, 50) if col == "#0f172a" else np.linspace(CUT, CUT + 12, 50)
    ax.plot(gx, np.polyval(np.polyfit(x[m], y[m], 1), gx), color=col, lw=2.5)
tau_all = local_linear(x, w, y, 8.0)
_band(ax)
ax.set_title(f"All data:  $\\hat\\tau$ = {tau_all:.1f}  (truth: {TAU:.0f})", fontsize=10, color="#b91c1c")
ax.set_xlabel("Eligibility score  X"); ax.set_ylabel("Binned mean of  Y")

ax = axes[1]                                     # donut: drop |x-c| < R and refit
keep = np.abs(x - CUT) >= R
_binned_range(ax, x[keep], y[keep], 0, CUT - R, CTRLC)
_binned_range(ax, x[keep], y[keep], CUT + R, 100, ACCENT)
ax.axvspan(CUT - R, CUT + R, color="#fca5a5", alpha=.25, zorder=1)
for side_mask, col in (((x < CUT) & keep, "#0f172a"), ((x >= CUT) & keep, ACCENT2)):
    m = side_mask & (np.abs(x - CUT) <= 12 + R)
    gx = np.linspace(CUT - 15, CUT, 50) if col == "#0f172a" else np.linspace(CUT, CUT + 15, 50)
    ax.plot(gx, np.polyval(np.polyfit(x[m], y[m], 1), gx), color=col, lw=2.5)
tau_donut = local_linear(x[keep], w[keep], y[keep], 8.0 + R)
_band(ax)
ax.set_title(f"Donut $\\pm${R:.0f}:  $\\hat\\tau$ = {tau_donut:.1f}", fontsize=10, color=ACCENT2)
ax.set_xlabel("Eligibility score  X")
_save(fig, "RD_Figure_8.png")